# Deepfake Audio Detection System - Integrated Edition
### CPU-Optimized & Fully Self-Contained

This notebook contains the complete project logic integrated into cells. No external script calls are required.

---

## 1. Setup Environment
Install necessary system dependencies and Python packages.

In [ ]:
# Install system dependencies
!apt-get update -qq
!apt-get install -y -qq espeak-ng ffmpeg libportaudio2

# Install Python packages
!pip install -q torch>=2.0.0 torchaudio>=2.0.0 transformers>=4.35.0 datasets>=2.14.0 accelerate>=0.24.0
!pip install -q librosa>=0.10.0 soundfile>=0.12.1 pydub>=0.25.1 noisereduce>=3.0.0
!pip install -q numpy>=1.24.0 pandas>=2.0.0 scikit-learn>=1.3.0 scipy>=1.11.0
!pip install -q matplotlib>=3.7.0 seaborn>=0.12.0 tqdm>=4.66.0 tensorboard>=2.14.0
!pip install -q click>=8.1.0 rich>=13.6.0 pyyaml>=6.0.1 python-dotenv>=1.0.0 joblib>=1.3.0 pyttsx3>=2.90 gtts>=2.3.2 opendatasets

## 2. Global Configuration
Define all hyperparameters and directory structures.

In [ ]:
import os
from pathlib import Path
import torch
from google.colab import files

class Config:
    BASE_DIR      = Path("/content").resolve()
    DATA_DIR      = BASE_DIR / "data"
    RAW_DIR       = DATA_DIR / "raw"
    PROCESSED_DIR = DATA_DIR / "processed"
    CHECKPOINTS   = BASE_DIR / "checkpoints"
    RESULTS_DIR   = BASE_DIR / "results"
    
    REAL_RAW      = RAW_DIR / "real"
    FAKE_RAW      = RAW_DIR / "fake"
    REAL_PROC     = PROCESSED_DIR / "real"
    FAKE_PROC     = PROCESSED_DIR / "fake"

    SAMPLE_RATE      = 16000
    CHUNK_DURATION   = 4.0
    MODEL_NAME       = "facebook/wav2vec2-base"
    
    # Training
    BATCH_SIZE     = 4  # Reduced for CPU
    NUM_EPOCHS     = 1  # Reduced for CPU demo
    LEARNING_RATE  = 2e-5
    FP16           = False # Must be False for CPU
    DEVICE         = "cuda" if torch.cuda.is_available() else "cpu"

cfg = Config()
for p in [cfg.REAL_RAW, cfg.FAKE_RAW, cfg.REAL_PROC, cfg.FAKE_PROC, cfg.CHECKPOINTS, cfg.RESULTS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print(f"Using device: {cfg.DEVICE}")

## 3. Core Components (Model & Dataset)
Integrated Wav2Vec2 detector and custom dataset loader.

In [ ]:
import nn_module
import torch.nn as nn
from transformers import Wav2Vec2Config, Wav2Vec2Model, Wav2Vec2PreTrainedModel, Wav2Vec2FeatureExtractor
from transformers.modeling_outputs import SequenceClassifierOutput
import librosa, numpy as np, pandas as pd, soundfile as sf
from torch.utils.data import DataLoader, Dataset

class ClassificationHead(nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        self.net = nn.Sequential(
            nn.LayerNorm(hidden_size),
            nn.Linear(hidden_size, 256),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(256, 2)
        )
    def forward(self, x): return self.net(x)

class DeepfakeDetector(Wav2Vec2PreTrainedModel):
    def __init__(self, config):
        super().__init__(config)
        self.wav2vec2 = Wav2Vec2Model(config)
        self.classifier = ClassificationHead(config.hidden_size)
        self.init_weights()
    def forward(self, input_values, attention_mask=None, labels=None, **kwargs):
        outputs = self.wav2vec2(input_values, attention_mask=attention_mask)
        pooled = outputs.last_hidden_state.mean(dim=1)
        logits = self.classifier(pooled)
        loss = nn.CrossEntropyLoss()(logits, labels) if labels is not None else None
        return SequenceClassifierOutput(loss=loss, logits=logits)

class DeepfakeDataset(Dataset):
    def __init__(self, records, fe):
        self.records, self.fe = records, fe
        self.max_s = int(cfg.CHUNK_DURATION * cfg.SAMPLE_RATE)
    def __len__(self): return len(self.records)
    def __getitem__(self, idx):
        r = self.records[idx]
        y, _ = librosa.load(r['path'], sr=cfg.SAMPLE_RATE)
        if len(y) > self.max_s: y = y[:self.max_s]
        else: y = np.pad(y, (0, self.max_s - len(y)))
        e = self.fe(y, sampling_rate=cfg.SAMPLE_RATE, return_tensors='pt', padding='max_length', max_length=self.max_s)
        return {'input_values': e.input_values.squeeze(0), 'attention_mask': e.attention_mask.squeeze(0), 'labels': torch.tensor(r['label_id'])}

def get_dataloaders(fe):
    df = pd.read_csv(cfg.PROCESSED_DIR / "metadata.csv")
    recs = df.to_dict('records')
    from sklearn.model_selection import train_test_split
    tr_recs, val_recs = train_test_split(recs, test_size=0.15, random_state=42)
    return DataLoader(DeepfakeDataset(tr_recs, fe), batch_size=cfg.BATCH_SIZE, shuffle=True), DataLoader(DeepfakeDataset(val_recs, fe), batch_size=cfg.BATCH_SIZE)

## 4. Pipeline Engine
Functions for data generation, preprocessing, and training.

In [ ]:
from tqdm.notebook import tqdm
import pyttsx3, random

def generate_synthetic_data(n=100):
    """Generate fake audio to match the real dataset count."""
    print(f"Generating {n} synthetic samples...")
    engine = pyttsx3.init()
    prompts = ["Testing synthetic voice.", "This is a deepfake detector validation.", "AI audio sample for training."]
    for i in tqdm(range(n), desc="Synthetic Gen"):
        p = random.choice(prompts)
        engine.save_to_file(p, str(cfg.FAKE_RAW / f"fake_{i:04d}.wav"))
        engine.runAndWait()

def run_preprocessing():
    """Normalize and build metadata."""
    print("Preprocessing audio files...")
    manifest = []
    for label in ['real', 'fake']:
        src = cfg.RAW_DIR / label
        dst = cfg.PROCESSED_DIR / label
        dst.mkdir(parents=True, exist_ok=True)
        files = list(src.rglob("*.wav"))
        for f in tqdm(files, desc=f"Prep {label}"):
            try:
                y, _ = librosa.load(f, sr=cfg.SAMPLE_RATE)
                out_p = dst / f.name
                sf.write(out_p, y, cfg.SAMPLE_RATE)
                manifest.append({'path': str(out_p), 'label': label, 'label_id': 1 if label=='fake' else 0})
            except Exception: continue
    pd.DataFrame(manifest).to_csv(cfg.PROCESSED_DIR / "metadata.csv", index=False)

def train_model():
    """Run the fine-tuning loop."""
    fe = Wav2Vec2FeatureExtractor.from_pretrained(cfg.MODEL_NAME)
    tr_loader, val_loader = get_dataloaders(fe)
    config = Wav2Vec2Config.from_pretrained(cfg.MODEL_NAME, num_labels=2)
    mdl = DeepfakeDetector.from_pretrained(cfg.MODEL_NAME, config=config, ignore_mismatched_sizes=True).to(cfg.DEVICE)
    opt = torch.optim.AdamW(mdl.parameters(), lr=cfg.LEARNING_RATE)
    
    for epoch in range(cfg.NUM_EPOCHS):
        mdl.train()
        for batch in tqdm(tr_loader, desc=f"Epoch {epoch+1}"):
            v, m, l = batch['input_values'].to(cfg.DEVICE), batch['attention_mask'].to(cfg.DEVICE), batch['labels'].to(cfg.DEVICE)
            opt.zero_grad()
            loss = mdl(v, m, labels=l).loss
            loss.backward()
            opt.step()
    mdl.save_pretrained(cfg.CHECKPOINTS / "best_model")
    print("Training Finished. Model saved to checkpoints/best_model")

## 5. Main Loop (Dataset -> Train)
One-click execution of the entire pipeline.

In [ ]:
import opendatasets as od, shutil

# 1. Download Real Data
dataset_url = "https://www.kaggle.com/datasets/hmsolanki/indian-languages-audio-dataset"
od.download(dataset_url)
for wav in Path("indian-languages-audio-dataset").rglob("*.wav"): shutil.copy(wav, cfg.REAL_RAW / wav.name)

# 2. Generate Fake Data
n_real = len(list(cfg.REAL_RAW.glob("*.wav")))
generate_synthetic_data(n=n_real)

# 3. Preprocess
run_preprocessing()

# 4. Train
train_model()

## 6. Inference Loop
Test on the processed data.

In [ ]:
def predict_audio(path):
    mdl = DeepfakeDetector.from_pretrained(cfg.CHECKPOINTS / "best_model").to(cfg.DEVICE).eval()
    fe = Wav2Vec2FeatureExtractor.from_pretrained(cfg.MODEL_NAME)
    y, _ = librosa.load(path, sr=cfg.SAMPLE_RATE)
    v = fe(y, sampling_rate=cfg.SAMPLE_RATE, return_tensors='pt').input_values.to(cfg.DEVICE)
    with torch.no_grad(): out = mdl(v).logits; prob = torch.softmax(out, dim=-1)
    return 'fake' if prob[0,1] > 0.5 else 'real'

print("Ready for inference. Use predict_audio('path_to_wav')")